# 04 — Validação da Rede, Diagnósticos Pós-Treino e Testes de Sanidade

Este notebook carrega os modelos salvos em `outputs/checkpoints/` (**sejam eles treinados no CERN HTCondor via `scripts/train.py` ou no `03_training.ipynb`**) e executa toda a bateria de validação técnica e física:

1. **Curvas de aprendizado e Loss estilo Fig. 50 da AN-25-103** (treino vs validação sem oversampling, ponderados por `val_freq_w`).
2. **Teste de Overtraining (Kolmogorov-Smirnov)** nas distribuições de probabilidade por classe.
3. **Diagnóstico profundo da validação (6 baterias):** concentração de `val_freq_w`, caudas de Cross-Entropy nos logits e saturação, BatchNorm móvel vs mediana de batches, outliers na saída do LBN, memorização nos pools mais pesados (`CE_val / CE_train`) e detecção de épocas anômalas.
4. **Auditoria do condicionamento em $\kappa_\lambda$:** dependência de $\langle P(\text{HH})\rangle$, **Matriz de Resposta $\kappa_{\text{verdadeiro}} \times \kappa_{\text{entrada}}$** (com controle cego na `DNN_AN`) e visualização dos vetores aprendidos $s(\kappa_\lambda)$ e $b(\kappa_\lambda)$ nos 8 blocos `AffineConditioning`.
5. **Estudo de *Mass Sculpting* (AN-25-103 Seção 5.5)** nas fatias de score da rede.
6. **Testes opcionais de sanidade (13.4 Random-Label, 13.5 Interpolação em $\kappa_\lambda = 2.45$, 13.6 Seeds)** — execução sob demanda ou leitura das tabelas geradas pelo HTCondor.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data.preprocessing import make_inputs, prepare_datasets_from_config
from src.data.weights import oversampling_frequency_weights
from src.evaluation.kappa_scan import (
    interpolation_study,
    kappa_dependence,
    kappa_response_matrix,
    seed_study,
)
from src.evaluation.plotting import (
    plot_affine_conditioning_params,
    plot_interpolation_study,
    plot_kappa_dependence,
    plot_kappa_response_heatmap,
    plot_mass_sculpting,
    plot_overtraining_distributions,
    plot_random_label_test,
    plot_seed_stability,
    plot_training_history,
)
from src.evaluation.score_distributions import (
    mass_sculpting_slices,
    overtraining_ks,
    reconstruct_invariant_masses,
)
from src.models.affine_dnn import extract_affine_modulation
from src.training.diagnostics import random_label_test, run_validation_diagnostics
from src.training.losses import an_style_loss
from src.training.train import create_generator_for_model, load_model_bundle
from src.utils.config import load_config
from src.utils.paths import OutputPaths
from src.utils.seed import configure_gpu_memory_growth, set_all_seeds

configure_gpu_memory_growth()
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

## 1. Carregamento dos Checkpoints (`Condor` ou `Notebook 03`) e Dados

In [ ]:
# Aponte CHECKPOINTS_DIR para a pasta no EOS se treinou via HTCondor em outro diretório:
CUSTOM_OUTPUT_DIR = None   # Ex.: "/eos/user/m/mgeralde/Analysis_hh2bbtautau/PNN/affinepnn_outputs"
MODEL_NAMES = ["AffinePNN", "DNN_AN"]

cfg = load_config(PROJECT_ROOT / "configs" / "default.yaml")
if CUSTOM_OUTPUT_DIR:
    cfg["paths"]["output_dir"] = CUSTOM_OUTPUT_DIR

set_all_seeds(cfg["seed"])
out_paths = OutputPaths.from_root(cfg["paths"]["output_dir"], create=True)
ckpt_dir = out_paths.checkpoints

models = {}
histories = {}
metas = {}
for name in MODEL_NAMES:
    m, h, meta = load_model_bundle(ckpt_dir, name)
    models[name] = m
    if h is not None:
        histories[name] = h
    if meta is not None:
        metas[name] = meta
    print(f"Modelo carregado: {name:10s} | params = {m.count_params():,d}")

data = prepare_datasets_from_config(cfg, existing_checkpoints_dir=ckpt_dir, verbose=True)

## 2. Histórico de Treinamento, Loss Estilo Fig. 50 da AN e Teste KS de Overtraining

In [ ]:
if histories:
    plot_training_history(histories, save_path=out_paths.plots / "val_training_history.png", show=True)

knorm = float(cfg["physics"]["kappa_norm"])
eval_bs = int(cfg["evaluation"]["eval_batch_size"])
val_df = data.splits["val"]
val_y = val_df["label"].to_numpy()
val_inputs = make_inputs(data.x_cont["val"], data.x_cat["val"], val_df["kappa_eval"].to_numpy(), kappa_norm=knorm)

gen_ref = create_generator_for_model(cfg, "affine", data)
val_freq_w = oversampling_frequency_weights(
    val_df,
    gen_ref,
    class_fraction=cfg["sampling"]["class_batch_fraction"],
    clip=cfg["sampling"]["val_freq_clip"],
)

fig50_df = an_style_loss(
    models=models,
    train_df=data.splits["train"],
    x_cont_train=data.x_cont["train"],
    x_cat_train=data.x_cat["train"],
    val_inputs=val_inputs,
    val_y=val_y,
    val_freq_w=val_freq_w,
    gen_ref=gen_ref,
    kappa_norm=knorm,
    class_fraction=cfg["sampling"]["class_batch_fraction"],
    val_freq_clip=cfg["sampling"]["val_freq_clip"],
    batch_size=eval_bs,
    seed=cfg["seed"],
)
print("Loss ponderada pela frequência do oversampling (comparável treino x validação, Fig. 50 da AN):")
display(fig50_df)

for name, m in models.items():
    print(f"\nTeste Kolmogorov-Smirnov e Distribuições de Overtraining (treino vs val) — {name}:")
    ks_df, ks_scores = overtraining_ks(
        m,
        data.splits["train"],
        data.x_cont["train"],
        data.x_cat["train"],
        val_inputs,
        val_y,
        cfg["physics"]["class_labels"],
        kappa_norm=knorm,
        batch_size=eval_bs,
        seed=cfg["seed"],
        return_scores=True,
    )
    display(ks_df)
    plot_overtraining_distributions(
        p_train=ks_scores["p_train"],
        y_train=ks_scores["y_train"],
        p_eval=ks_scores["p_eval"],
        y_eval=ks_scores["y_eval"],
        class_labels=cfg["physics"]["class_labels"],
        model_name=name,
        split_label="val",
        save_path=out_paths.plots / f"val_overtraining_distributions_{name}.png",
        show=True,
    )

## 3. Diagnóstico Profundo Pós-Treino (Caudas de CE, BatchNorm, Outliers do LBN e Memorização por Pool)

In [ ]:
diag_results = run_validation_diagnostics(
    models=models,
    histories=histories,
    data=data,
    val_inputs=val_inputs,
    val_freq_w=val_freq_w,
    gen_ref=gen_ref,
    continuous_features=cfg["features"]["continuous_features"],
    class_labels=cfg["physics"]["class_labels"],
    kappa_norm=knorm,
    eval_batch_size=eval_bs,
    verbose=True,
)

## 4. Auditoria da Parametrização em $\kappa_\lambda$ e Camadas `AffineConditioning`

- Para a **`AffinePNN`**, a diagonal da matriz $\kappa_{\text{verdadeiro}} \times \kappa_{\text{entrada}}$ deve apresentar os melhores valores de $\text{AUC}_w$ de cada linha.
- Para a **`DNN_AN`** (controle Single-NN), todas as colunas devem ser **exatamente idênticas** ($\partial L / \partial \kappa_\lambda = 0$).

In [ ]:
kgrid = cfg["physics"]["kappa_grid"]
matrices = {}

for name, m in models.items():
    dep_df, std_by_cls = kappa_dependence(
        m, val_df, data.x_cont["val"], data.x_cat["val"],
        cfg["physics"]["class_labels"], kappas=kgrid, kappa_norm=knorm, batch_size=eval_bs, seed=cfg["seed"]
    )
    print(f"Desvio-padrão médio entre hipóteses de κ_λ ({name}):", std_by_cls)
    plot_kappa_dependence(dep_df, cfg["physics"]["class_labels"], model_name=name, show=True)
    matrices[name] = kappa_response_matrix(
        m, val_df, data.x_cont["val"], data.x_cat["val"], kappas=kgrid, kappa_norm=knorm, batch_size=eval_bs
    )
    print(f"\nMatriz de Resposta AUC_w ({name}):")
    display(matrices[name].round(4))

plot_kappa_response_heatmap(
    matrices,
    save_path=out_paths.plots / "val_kappa_response_heatmap.png",
    show=True,
)

if "AffinePNN" in models:
    scan_k = np.linspace(-2.0, 8.0, 41)
    mod_dict = extract_affine_modulation(models["AffinePNN"], scan_k, kappa_norm=knorm)
    plot_affine_conditioning_params(
        mod_dict,
        save_path=out_paths.plots / "val_affine_conditioning_modulation.png",
        show=True,
    )

## 5. Checagem de *Mass Sculpting* no Fundo (AN-25-103 Seção 5.5)

Inspeciona a distribuição de $m_{\tau\tau}$, $m_{bb}$ e $m_{HH}$ dos eventos de fundo em fatias crescentes de $P(\text{HH})$.

In [ ]:
masses_val = reconstruct_invariant_masses(val_df)
for name, m in models.items():
    p_val = m.predict(val_inputs, batch_size=eval_bs, verbose=0)[:, 2]
    sculpt_df = mass_sculpting_slices(val_df, p_val)
    print(f"Estatísticas de massa por fatia de score — {name}:")
    display(sculpt_df.head(12))
    for mcol, max_m in (("m_tautau", 260.0), ("m_bb", 280.0), ("m_hh", 850.0)):
        plot_mass_sculpting(
            val_df,
            masses_val,
            p_val,
            mass_col=mcol,
            model_name=name,
            max_mass=max_m,
            save_path=out_paths.plots / f"mass_sculpting_{mcol}_{name}.png",
            show=True,
        )

## 6. Estudos de Sanidade Opcionais (13.4 Random-Label, 13.5 Interpolação, 13.6 Múltiplas Seeds)

Estes estudos treinam múltiplos modelos do zero (recomendado submeter via `condor_submit condor/sanity_studies.sub`). Se os CSVs já tiverem sido gerados pelo HTCondor em `outputs/metrics/`, a célula abaixo os carrega e exibe automaticamente!

In [ ]:
RUN_RANDOM_LABEL_NOW = False
RUN_INTERPOLATION_NOW = False
RUN_SEEDS_NOW = False

# 13.4 Random-label test
rl_csv = out_paths.metrics / "sanity_random_label_test.csv"
if RUN_RANDOM_LABEL_NOW:
    rl_df = random_label_test(cfg, data, val_inputs, seeds=cfg["studies"]["random_label_seeds"], epochs=cfg["studies"]["random_label_epochs"])
    rl_df.to_csv(rl_csv, index=False)
    plot_random_label_test(rl_df, save_path=out_paths.plots / "sanity_random_label_test.png", show=True)
elif rl_csv.exists():
    print("Carregado resultado salvo do Random-Label Test (13.4):")
    rl_df = pd.read_csv(rl_csv)
    display(rl_df)
    plot_random_label_test(rl_df, save_path=out_paths.plots / "sanity_random_label_test.png", show=True)
else:
    print("13.4 Random-Label não executado aqui (rode via `scripts/sanity_studies.py --study random_label` ou ligue RUN_RANDOM_LABEL_NOW=True).")

# 13.5 Interpolation study (held-out κ_λ = 2.45)
interp_csv = out_paths.metrics / "sanity_interpolation_kl2p45.csv"
if RUN_INTERPOLATION_NOW:
    interp_ser = interpolation_study(
        cfg,
        data,
        val_freq_w,
        model_full_ref=models.get("AffinePNN"),
        model_dnn_ref=models.get("DNN_AN"),
        held_out=2.45,
    )
    interp_ser.to_frame().to_csv(interp_csv)
    plot_interpolation_study(interp_ser, held_out=2.45, save_path=out_paths.plots / "sanity_interpolation_kl2p45.png", show=True)
elif interp_csv.exists():
    print("\nCarregado resultado salvo do Estudo de Interpolação (13.5):")
    interp_df = pd.read_csv(interp_csv)
    display(interp_df)
    plot_interpolation_study(interp_df, held_out=2.45, save_path=out_paths.plots / "sanity_interpolation_kl2p45.png", show=True)

# 13.6 Multi-seed study (agrega arquivos gerados por job array do HTCondor se existirem)
seed_files = sorted(out_paths.metrics.glob("sanity_seeds_metrics_*.csv"))
if RUN_SEEDS_NOW:
    seeds_df, gain_df = seed_study(cfg, data, seeds=cfg["studies"]["seeds_for_study"])
    plot_seed_stability(seeds_df, save_path=out_paths.plots / "sanity_seeds_stability.png", show=True)
elif seed_files:
    all_seeds_df = pd.concat([pd.read_csv(f) for f in seed_files], ignore_index=True).drop_duplicates(subset=["modelo", "kappa", "seed"])
    print(f"\nCarregados {len(seed_files)} arquivo(s) do Estudo de Seeds do Condor:")
    display(all_seeds_df.groupby(["modelo", "kappa"])["auc_w"].agg(["count", "mean", "std"]))
    piv = all_seeds_df.pivot_table(index=["kappa", "seed"], columns="modelo", values="auc_w")
    if {"AffinePNN", "DNN_AN"}.issubset(piv.columns):
        print("Ganho médio AffinePNN - DNN_AN entre seeds:")
        display((piv["AffinePNN"] - piv["DNN_AN"]).groupby("kappa").agg(["mean", "std"]))
    plot_seed_stability(all_seeds_df, save_path=out_paths.plots / "sanity_seeds_stability.png", show=True)